# Workspace Building

This notebook demonstrates how to build a serialized workspace from a YAML fit configuration file.
The workspace is a JSON dictionary (following a pyhf-like schema) that contains all the information
needed to construct the statistical model: binned histograms, density ratios, systematic variations,
normalization factors, and measurement definitions.

Once built, the workspace is saved to disk as JSON and can be loaded independently for fitting
in `parameter_fitting.ipynb` — no config file or ROOT files needed at fit time.

In [ ]:
import os
import pprint
import nsbi_common_utils

## Configuration

Point to your fit configuration YAML files. These define samples, regions, systematics,
and paths to ROOT files and density ratio `.npy` files.

In [ ]:
hist_config_path = "./config_fit_histogram.yml"
nsbi_config_path = "./config_fit_nsbi.yml"

output_dir = "./workspaces"
os.makedirs(output_dir, exist_ok=True)

## Build histogram workspace

The histogram workspace contains only binned channels (control regions, binned signal region).
This is the traditional HistFactory-style analysis.

In [ ]:
builder_hist = nsbi_common_utils.workspace_builder.WorkspaceBuilder(config_path=hist_config_path)
ws_hist = builder_hist.build()
builder_hist.dump_workspace(ws_hist, os.path.join(output_dir, "workspace_histogram.json"))

## Build NSBI workspace

The NSBI workspace extends the histogram workspace with unbinned channels that use
neural density ratios. It references pre-computed `.npy` ratio files and Asimov weights.

In [ ]:
builder_nsbi = nsbi_common_utils.workspace_builder.WorkspaceBuilder(config_path=nsbi_config_path)
ws_nsbi = builder_nsbi.build()
builder_nsbi.dump_workspace(ws_nsbi, os.path.join(output_dir, "workspace_nsbi.json"))

## Inspect the workspace

The workspace is a dictionary with `channels`, `measurements`, and `version` keys.
Each channel contains samples with nominal data, modifiers (normfactors, systematics),
and for unbinned channels, paths to density ratio arrays.

In [ ]:
print("NSBI workspace keys:", list(ws_nsbi.keys()))
print(f"\nNumber of channels: {len(ws_nsbi['channels'])}")
for ch in ws_nsbi['channels']:
    print(f"  - {ch['name']} (type: {ch.get('type', 'binned')}, samples: {[s['name'] for s in ch['samples']]})")
print(f"\nMeasurement: {ws_nsbi['measurements'][0]['name']}")
print(f"POI: {ws_nsbi['measurements'][0]['config']['poi']}")
print(f"Parameters: {[p['name'] for p in ws_nsbi['measurements'][0]['config']['parameters']]}")

In [ ]:
print("Full NSBI workspace:\n")
pprint.pprint(ws_nsbi)

## Next steps

The saved workspace JSON files can now be loaded in `parameter_fitting.ipynb` for
model construction and hypothesis testing — without needing the original config files
or ROOT files at fit time.

```python
ws = nsbi_common_utils.workspace_builder.WorkspaceBuilder.load_workspace("workspaces/workspace_nsbi.json")
model = nsbi_common_utils.models.sbi_parametric_model(workspace=ws, measurement_to_fit="my_measurement")
```